# Scene 2 — Full-Frame Kamishibai Layer Pipeline (Colab)

This notebook builds an end-to-end pipeline:
1. Detect semantic objects (Florence-equivalent zero-shot detector: **OWL-V2**)
2. Segment each detected object with **SAM**
3. Estimate depth and assign `background/midground/foreground`
4. Keep **full-frame masks** (no cropping)
5. Generate **full-frame stylized layers** with OpenAI Image Edit API
6. Save masks, layers, transparent variants, and metadata JSON
7. Visualize boxes, masks, layers, and stacked preview

> **Constraint respected:** the OpenAI call always uses full-frame image + full-frame mask.


In [ ]:
# Colab setup
!pip -q install --upgrade pip
!pip -q install openai pillow numpy matplotlib opencv-python tqdm transformers accelerate timm scipy


In [ ]:
import os
import io
import json
import math
import base64
from typing import List, Dict

import cv2
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from transformers import pipeline, Owlv2Processor, Owlv2ForObjectDetection, SamProcessor, SamModel
from openai import OpenAI

print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
# =========================
# Config
# =========================
INPUT_IMAGE_PATH = '/content/data/input/input.png'
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
assert os.path.exists(INPUT_IMAGE_PATH), f'Missing input image: {INPUT_IMAGE_PATH}'
assert OPENAI_API_KEY, 'OPENAI_API_KEY is not set.'

DETECTION_QUERIES = [
    'person', 'man', 'woman', 'child', 'deer', 'animal', 'dog', 'cat',
    'tree', 'forest', 'building', 'house', 'temple', 'sky', 'cloud', 'mountain',
    'ground', 'grass', 'road', 'water', 'river'
]
DETECTION_SCORE_THRESHOLD = 0.15
NMS_IOU_THRESHOLD = 0.45
MIN_MASK_AREA_FRAC = 0.001

OPENAI_IMAGE_MODEL = 'gpt-image-1'

# Exact fixed prompt (same for every layer)
STYLE_PROMPT = (
    'Japanese paper theater (kamishibai) paper-cut style illustration. '
    'Flat layered colors, simplified silhouette shapes, clean cut-paper edges, '
    'minimal texture, no photorealism, no gradients except subtle paper shading, '
    'limited cohesive palette, high contrast readable forms. '
    'Render ONLY the masked subject as the main element; keep everything else simple and unobtrusive. '
    'Preserve original full-frame composition and scale. Output must be full-frame.'
)

ROOT = '/content/data'
DIR_INPUT = os.path.join(ROOT, 'input')
DIR_MASKS = os.path.join(ROOT, 'masks')
DIR_LAYERS = os.path.join(ROOT, 'layers')
DIR_DEBUG = os.path.join(ROOT, 'debug')
for d in [DIR_INPUT, DIR_MASKS, DIR_LAYERS, DIR_DEBUG]:
    os.makedirs(d, exist_ok=True)

print('Directories ready:', DIR_INPUT, DIR_MASKS, DIR_LAYERS, DIR_DEBUG)
print('Using style prompt:', STYLE_PROMPT)


In [ ]:
# =========================
# Utilities
# =========================
def load_image_rgb(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('RGB'))

def save_png(arr: np.ndarray, path: str):
    Image.fromarray(arr).save(path)

def clamp_box_xyxy(box, w, h):
    x1, y1, x2, y2 = box
    x1 = int(max(0, min(w - 1, x1)))
    y1 = int(max(0, min(h - 1, y1)))
    x2 = int(max(0, min(w - 1, x2)))
    y2 = int(max(0, min(h - 1, y2)))
    if x2 <= x1: x2 = min(w - 1, x1 + 1)
    if y2 <= y1: y2 = min(h - 1, y1 + 1)
    return [x1, y1, x2, y2]

def nms_xyxy(boxes, scores, iou_thr=0.45):
    if len(boxes) == 0:
        return []
    boxes = np.array(boxes, dtype=np.float32)
    scores = np.array(scores, dtype=np.float32)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(boxes[i, 0], boxes[order[1:], 0])
        yy1 = np.maximum(boxes[i, 1], boxes[order[1:], 1])
        xx2 = np.minimum(boxes[i, 2], boxes[order[1:], 2])
        yy2 = np.minimum(boxes[i, 3], boxes[order[1:], 3])
        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        area_j = (boxes[order[1:], 2] - boxes[order[1:], 0]) * (boxes[order[1:], 3] - boxes[order[1:], 1])
        iou = inter / (area_i + area_j - inter + 1e-6)
        inds = np.where(iou <= iou_thr)[0]
        order = order[inds + 1]
    return keep

def clean_binary_mask(mask: np.ndarray, min_area: int) -> np.ndarray:
    m = (mask > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    cleaned = np.zeros_like(m)
    for i in range(1, num):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            cleaned[labels == i] = 1
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
    cleaned = cv2.GaussianBlur((cleaned * 255).astype(np.uint8), (0, 0), sigmaX=1.0)
    return (cleaned > 127).astype(np.uint8)

def mask_to_rgba_cutout(full_rgb: np.ndarray, mask01: np.ndarray) -> np.ndarray:
    return np.dstack([full_rgb, (mask01 * 255).astype(np.uint8)])

def draw_boxes(image_rgb: np.ndarray, dets: List[Dict]) -> np.ndarray:
    img = Image.fromarray(image_rgb.copy())
    draw = ImageDraw.Draw(img)
    for d in dets:
        x1, y1, x2, y2 = d['bbox']
        draw.rectangle([x1, y1, x2, y2], outline=(255, 50, 50), width=3)
        draw.text((x1 + 3, max(0, y1 - 14)), f"{d['label']}:{d['score']:.2f}", fill=(255, 50, 50))
    return np.array(img)


In [ ]:
# 1) Object detection (Florence equivalent: OWL-V2 zero-shot)
def detect_objects_owlv2(image_rgb: np.ndarray, queries: List[str], score_thr=0.15):
    processor = Owlv2Processor.from_pretrained('google/owlv2-base-patch16-ensemble')
    model = Owlv2ForObjectDetection.from_pretrained('google/owlv2-base-patch16-ensemble').to(DEVICE)

    pil = Image.fromarray(image_rgb)
    inputs = processor(text=[queries], images=pil, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([pil.size[::-1]], device=DEVICE)
    results = processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=score_thr)[0]

    dets = []
    h, w = image_rgb.shape[:2]
    for box, score, label in zip(results['boxes'], results['scores'], results['labels']):
        b = clamp_box_xyxy(box.detach().cpu().numpy().tolist(), w, h)
        dets.append({'label': queries[int(label.item())], 'score': float(score.item()), 'bbox': b})

    keep = nms_xyxy([d['bbox'] for d in dets], [d['score'] for d in dets], iou_thr=NMS_IOU_THRESHOLD)
    return [dets[i] for i in keep]


In [ ]:
# 2) Segmentation with SAM
def segment_with_sam(image_rgb: np.ndarray, detections: List[Dict]) -> List[Dict]:
    if len(detections) == 0:
        return []

    sam_processor = SamProcessor.from_pretrained('facebook/sam-vit-base')
    sam_model = SamModel.from_pretrained('facebook/sam-vit-base').to(DEVICE)

    pil = Image.fromarray(image_rgb)
    input_boxes = [[d['bbox'] for d in detections]]
    inputs = sam_processor(images=pil, input_boxes=input_boxes, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        outputs = sam_model(**inputs)

    masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs['original_sizes'].cpu(),
        inputs['reshaped_input_sizes'].cpu()
    )[0]

    h, w = image_rgb.shape[:2]
    min_area = int(h * w * MIN_MASK_AREA_FRAC)

    segmented = []
    for i, d in enumerate(detections):
        m_candidates = masks[i].numpy()
        ious = outputs.iou_scores[0, i].detach().cpu().numpy()
        best = int(np.argmax(ious))
        mask = (m_candidates[best] > 0).astype(np.uint8)
        mask = clean_binary_mask(mask, min_area=min_area)
        if mask.sum() < min_area:
            continue
        segmented.append({**d, 'mask': mask})
    return segmented


In [ ]:
# 3) Depth estimation + layer assignment
def estimate_depth_map(image_rgb: np.ndarray) -> np.ndarray:
    depth_pipe = pipeline(task='depth-estimation', model='Intel/dpt-large', device=0 if DEVICE=='cuda' else -1)
    out = depth_pipe(Image.fromarray(image_rgb))
    depth = np.array(out['depth']).astype(np.float32)
    depth = cv2.resize(depth, (image_rgb.shape[1], image_rgb.shape[0]), interpolation=cv2.INTER_CUBIC)
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    return depth

def assign_depth_layers(segmented: List[Dict], depth_map: np.ndarray) -> List[Dict]:
    obj_depths = []
    for s in segmented:
        vals = depth_map[s['mask'] > 0]
        obj_depths.append(float(np.median(vals)) if len(vals) else 0.5)

    order = np.argsort(obj_depths)
    n = len(segmented)
    out = [None] * n
    for rank, idx in enumerate(order):
        if rank < max(1, n // 3):
            layer = 'foreground'
        elif rank < max(2, 2 * n // 3):
            layer = 'midground'
        else:
            layer = 'background'
        s = dict(segmented[idx])
        s['depth_score'] = obj_depths[idx]
        s['layer'] = layer
        out[idx] = s
    return out


In [ ]:
# 4+5) Full-frame OpenAI layer generation (CRITICAL: no cropping)
client = OpenAI(api_key=OPENAI_API_KEY)

def build_fullframe_edit_assets(image_rgb: np.ndarray, object_mask01: np.ndarray):
    h, w = object_mask01.shape
    base = np.full((h, w, 3), 245, dtype=np.uint8)
    # transparent where editable (object); opaque elsewhere
    alpha = np.where(object_mask01 > 0, 0, 255).astype(np.uint8)
    rgba_mask = np.dstack([np.zeros((h, w, 3), dtype=np.uint8), alpha])
    return Image.fromarray(base, 'RGB'), Image.fromarray(rgba_mask, 'RGBA')

def openai_generate_layer_fullframe(image_rgb: np.ndarray, mask01: np.ndarray, prompt: str, model: str = OPENAI_IMAGE_MODEL):
    base_pil, mask_pil = build_fullframe_edit_assets(image_rgb, mask01)

    img_bytes = io.BytesIO(); base_pil.save(img_bytes, format='PNG'); img_bytes.seek(0)
    msk_bytes = io.BytesIO(); mask_pil.save(msk_bytes, format='PNG'); msk_bytes.seek(0)

    result = client.images.edit(
        model=model,
        image=img_bytes,
        mask=msk_bytes,
        prompt=prompt,
        size=f"{image_rgb.shape[1]}x{image_rgb.shape[0]}"
    )
    b64 = result.data[0].b64_json
    return np.array(Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB'))

# Provider swap point:
# def generate_layer_fullframe_provider(image_rgb, mask01, prompt, provider='openai'):
#     if provider == 'openai':
#         return openai_generate_layer_fullframe(image_rgb, mask01, prompt)
#     elif provider == 'gemini':
#         raise NotImplementedError('Add Gemini image edit call with same full-frame mask strategy')
#     else:
#         raise ValueError(provider)


In [ ]:
# 6+7) Run pipeline and save outputs
image_rgb = load_image_rgb(INPUT_IMAGE_PATH)
h, w = image_rgb.shape[:2]

detections = detect_objects_owlv2(image_rgb, DETECTION_QUERIES, score_thr=DETECTION_SCORE_THRESHOLD)
print(f'Detections: {len(detections)}')

segmented = segment_with_sam(image_rgb, detections)
print(f'Segmented objects: {len(segmented)}')

depth_map = estimate_depth_map(image_rgb)
objects = assign_depth_layers(segmented, depth_map)

metadata = {
    'input_image': INPUT_IMAGE_PATH,
    'image_size': {'width': w, 'height': h},
    'style_prompt': STYLE_PROMPT,
    'objects': []
}

for i, obj in enumerate(tqdm(objects, desc='Generating stylized layers')):
    label = obj['label'].replace(' ', '_')
    layer = obj['layer']
    bbox = obj['bbox']
    mask01 = obj['mask'].astype(np.uint8)

    mask_path = os.path.join(DIR_MASKS, f'{i:03d}_{label}_mask.png')
    layer_path = os.path.join(DIR_LAYERS, f'{i:03d}_{label}_{layer}.png')
    rgba_path = os.path.join(DIR_LAYERS, f'{i:03d}_{label}_{layer}_rgba.png')

    save_png((mask01 * 255).astype(np.uint8), mask_path)

    stylized = openai_generate_layer_fullframe(image_rgb, mask01, STYLE_PROMPT)
    save_png(stylized, layer_path)

    rgba = mask_to_rgba_cutout(stylized, mask01)
    save_png(rgba, rgba_path)

    metadata['objects'].append({
        'id': i,
        'label': obj['label'],
        'layer': layer,
        'bbox': bbox,
        'depth_score': obj['depth_score'],
        'mask_path': mask_path,
        'layer_image_path': layer_path,
        'rgba_path': rgba_path
    })

meta_path = os.path.join(ROOT, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved metadata:', meta_path)
print('Layers generated:', len(metadata['objects']))


In [ ]:
# 8+9) Visualization
boxed = draw_boxes(image_rgb, detections)
plt.figure(figsize=(10, 8)); plt.imshow(boxed); plt.axis('off'); plt.title('Detected Boxes'); plt.show()

n = len(metadata['objects'])
cols = 4
rows = max(1, math.ceil(n / cols))

plt.figure(figsize=(4*cols, 4*rows))
for i, o in enumerate(metadata['objects']):
    m = np.array(Image.open(o['mask_path']).convert('L'))
    plt.subplot(rows, cols, i + 1)
    plt.imshow(m, cmap='gray')
    plt.title(f"{o['id']} {o['label']} ({o['layer']})")
    plt.axis('off')
plt.tight_layout(); plt.show()

plt.figure(figsize=(4*cols, 4*rows))
for i, o in enumerate(metadata['objects']):
    lay = np.array(Image.open(o['layer_image_path']).convert('RGB'))
    plt.subplot(rows, cols, i + 1)
    plt.imshow(lay)
    plt.title(f"{o['id']} {o['label']} ({o['layer']})")
    plt.axis('off')
plt.tight_layout(); plt.show()

stack = np.full_like(image_rgb, 245)
order_map = {'background': 0, 'midground': 1, 'foreground': 2}
for o in sorted(metadata['objects'], key=lambda x: order_map.get(x['layer'], 1)):
    lay = np.array(Image.open(o['layer_image_path']).convert('RGB'))
    m = (np.array(Image.open(o['mask_path']).convert('L')) > 0)
    stack[m] = lay[m]

plt.figure(figsize=(10, 8)); plt.imshow(stack); plt.axis('off'); plt.title('Stacked Preview'); plt.show()

save_png(boxed, os.path.join(DIR_DEBUG, 'detected_boxes.png'))
save_png(stack, os.path.join(DIR_DEBUG, 'stacked_preview.png'))
print('Debug previews:', DIR_DEBUG)


## Model swap notes

- Detection: replace OWL-V2 with Florence-2 / GroundingDINO.
- Segmentation: replace `facebook/sam-vit-base` with SAM2/HQ-SAM.
- Depth: replace DPT with MiDaS/Depth-Anything.
- Generation: keep the same full-frame strategy and fixed `STYLE_PROMPT` for consistency.

**Output folders**
- `/data/input/`
- `/data/masks/`
- `/data/layers/`
- `/data/debug/`
